In [4]:
import pandas as pd

# Load the file into a Pandas "DataFrame" (think of it as a virtual Excel table)
df = pd.read_csv('cern_hr_data.csv')

# Print out the exact names of the columns in your dataset
print("Here are your columns:")
print(df.columns.tolist())

Here are your columns:
['job_title', 'experience_years', 'education_level', 'skills_count', 'industry', 'company_size', 'location', 'remote_work', 'certifications', 'salary']


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# 1. Clean the data: Drop any rows that have missing information
df = df.dropna()

# 2. Separate features (X) and the target variable we want to predict (y)
# We are selecting the most relevant columns to predict the salary
features = ['job_title', 'experience_years', 'education_level', 'skills_count', 'certifications']
X = df[features]
y = df['salary']

# 3. Handle text data (like 'job_title' and 'education_level') using One-Hot Encoding
# AI only understands numbers, so this translates text into a numerical format
categorical_features = ['job_title', 'education_level', 'certifications']
categorical_transformer = OneHotEncoder(handle_unknown='ignore')

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', categorical_transformer, categorical_features)
    ], remainder='passthrough')

# 4. Create the Machine Learning Pipeline (Random Forest)
model = Pipeline(steps=[('preprocessor', preprocessor),
                        ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))])

# 5. Split the data: 80% for training, 20% for testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 6. Train the model!
print("Training the AI model... (this might take a few seconds)")
model.fit(X_train, y_train)

# 7. Test the model's accuracy on the 20% of data it hasn't seen yet
score = model.score(X_test, y_test)
print(f"Model trained successfully! Accuracy score (R-squared): {score:.2f}")

# 8. Generate predictions for the entire dataset so we can visualize it in Power BI
df['AI_Predicted_Salary'] = model.predict(X).round(0)

# 9. Save the essential columns to a new CSV file
output_columns = ['job_title', 'experience_years', 'education_level', 'salary', 'AI_Predicted_Salary']
df[output_columns].to_csv('predicted_salaries.csv', index=False)
print("Success! Predictions saved to 'predicted_salaries.csv'.")

Training the AI model... (this might take a few seconds)
Model trained successfully! Accuracy score (R-squared): 0.31
Success! Predictions saved to 'predicted_salaries.csv'.
